[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/33_computer_vision/90_exercises.ipynb)

# Bài tập — Chương 33: Computer Vision (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/33_computer_vision

In [ ]:
import math

import numpy as np

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_shape_scenes, make_shapes

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True


def make_cnn(in_ch=1, n_classes=5):
    return nn.Sequential(nn.Conv2d(in_ch, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(64 * 7 * 7, n_classes))


def train(make_model, X, y, steps=1000, augment=None, seed=0):
    torch.manual_seed(seed)
    model = make_model().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        model.train()
        idx = torch.randint(0, len(X), (64,), generator=g)
        xb = X[idx] if augment is None else augment(X[idx], g)
        opt.zero_grad()
        F.cross_entropy(model(xb.to(device)), y[idx].to(device)).backward()
        opt.step()
    return model


@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return sum((model(X[i:i + 500].to(device)).argmax(1).cpu() == y[i:i + 500]).sum().item() for i in range(0, len(X), 500)) / len(X)


def random_shift_flip(x, g, max_shift=0.15):
    B = x.shape[0]
    shift = (torch.rand(2, B, generator=g) * 2 - 1) * max_shift * 2
    sx = torch.where(torch.rand(B, generator=g) < 0.5, -1.0, 1.0)
    zeros, ones = torch.zeros(B), torch.ones(B)
    theta = torch.stack([torch.stack([sx, zeros, shift[0]], 1), torch.stack([zeros, ones, shift[1]], 1)], 1)
    return F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), align_corners=False)


to_tensor = lambda X: torch.tensor(X[:, None], dtype=torch.float32)

## L0 — Nhớ khái niệm

**Câu 1.** (a) HWC và CHW; thứ tự kênh của OpenCV? (b) Điều kiện để một phép augmentation hợp lệ? (c) Linear probe và fine-tune khác nhau thế nào; khi nào chọn mỗi cách? (d) IoU, NMS, AP là gì? (e) Vì sao pixel accuracy không đủ để đánh giá phân đoạn?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một nhà máy train CNN phát hiện sản phẩm lỗi từ ảnh băng chuyền. Dữ liệu: 5 000 ảnh chụp trong **một tuần** ở **dây chuyền A**; chia train/test ngẫu nhiên → accuracy 99%. Khi lắp sang **dây chuyền B** (đèn khác, camera đặt hơi lệch), accuracy còn 70%. Nhóm dự định "thu thêm 5 000 ảnh dây chuyền A". Phân tích và đề xuất.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Hộp $A = [10, 10, 30, 30]$, $B = [20, 15, 40, 35]$ (xyxy). Tính diện tích giao `inter`, hợp `union`, `iou`.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
inter = None
union = None
iou = None
### END CODE HERE ###
print(inter, union, round(iou, 4))

In [ ]:
check_equal(inter, 150, "giao: [20, 15, 30, 30]")
check_equal(union, 650, "hợp = 400 + 400 − 150")
check_close(iou, 150 / 650, message="IoU")

**Câu 4.** Một lớp có 4 vật thể thật trên tập test. Detector trả 6 dự đoán đã xếp theo điểm giảm dần, đúng/sai (IoU ≥ 0.5): `TP, TP, FP, TP, FP, FP`. Tính precision và recall sau mỗi dự đoán, rồi AP (diện tích dưới đường PR sau khi làm precision không tăng: $\sum (r_{k} - r_{k-1})\cdot\max_{j\ge k} p_j$).

In [ ]:
### START CODE HERE ### (≈ 6 dòng)
tp = None
recall = None
precision = None
interp = None
recall_prev = None
ap = None
### END CODE HERE ###
print("recall:", recall, "| precision:", precision.round(3), "| AP =", round(ap, 4))

In [ ]:
check_close(recall, [0.25, 0.5, 0.5, 0.75, 0.75, 0.75], message="recall")
check_close(precision, [1, 1, 2 / 3, 3 / 4, 3 / 5, 3 / 6], message="precision")
check_close(ap, 0.25 * 1 + 0.25 * 1 + 0.25 * 0.75, message="AP = 0.6875 (vật thứ 4 không bao giờ được tìm thấy)")

## L3 — Code

**Bài 5.** Augmentation cho detection phải biến đổi **cả ảnh lẫn hộp**. Viết `hflip_with_boxes(image, boxes)`: lật ngang ảnh (H, W) và trả hộp tương ứng (xyxy, toạ độ pixel kiểu `[x1, x2)`: lật thành $x_1' = W - x_2$, $x_2' = W - x_1$). Kiểm tra bằng mask: hộp tính lại từ mask đã lật phải trùng hộp bạn tính.

In [ ]:
def hflip_with_boxes(image, boxes):
    ### START CODE HERE ### (≈ 4 dòng)
    return image, boxes
    ### END CODE HERE ###

In [ ]:
X5, T5, M5 = make_shape_scenes(20, seed=3)
images_ok, boxes_ok = True, True
for i in range(20):
    img_f, boxes_f = hflip_with_boxes(X5[i], T5[i]["boxes"])
    images_ok &= np.array_equal(img_f, X5[i][:, ::-1])
    mask_f = M5[i][:, ::-1]                                             # mask lật — "đáp án" độc lập với hàm của bạn
    for box, c in zip(boxes_f, T5[i]["labels"]):
        ys, xs = np.nonzero(mask_f == c + 1)
        same_class = [(xs.min(), ys.min(), xs.max() + 1, ys.max() + 1)] if (T5[i]["labels"] == c).sum() == 1 else None
        if same_class:
            boxes_ok &= np.allclose(box, same_class[0])
check_true(images_ok, "ảnh được lật ngang đúng")
check_true(boxes_ok, "mọi hộp lật trùng hộp tính lại từ mask đã lật")
check_close(hflip_with_boxes(np.zeros((64, 64)), np.array([[0.0, 5, 10, 20]]))[1], [[54, 5, 64, 20]], message="ví dụ: [0, 5, 10, 20] → [54, 5, 64, 20]")

**Bài 6.** Cài `nms(boxes, scores, iou_threshold)` (trả chỉ số các hộp giữ lại, theo thứ tự điểm giảm dần) và `box_iou`. So khớp `torchvision.ops.nms` trên 200 hộp ngẫu nhiên.

In [ ]:
def box_iou(a, b):
    ### START CODE HERE ### (≈ 7 dòng)
    return None
    ### END CODE HERE ###


def nms(boxes, scores, iou_threshold=0.5):
    ### START CODE HERE ### (≈ 7 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
r6 = np.random.default_rng(0)
xy = r6.uniform(0, 100, (200, 2))
boxes6 = np.c_[xy, xy + r6.uniform(5, 30, (200, 2))]
scores6 = r6.random(200)
check_close(box_iou(boxes6[:3], boxes6[:3]).diagonal(), [1, 1, 1], message="IoU với chính nó = 1")
try:
    from torchvision.ops import nms as tv_nms
    ref6 = tv_nms(torch.tensor(boxes6), torch.tensor(scores6), 0.4).numpy()
    check_true(np.array_equal(nms(boxes6, scores6, 0.4), ref6), "khớp torchvision.ops.nms")
except ImportError:
    print("(không có torchvision — bỏ qua so sánh)")
kept = nms(boxes6, scores6, 0.4)
check_less(box_iou(boxes6[kept], boxes6[kept])[~np.eye(len(kept), dtype=bool)].max(), 0.4, "các hộp giữ lại chồng nhau < ngưỡng")

## L4 — Debug

**Bài 7.** Ảnh màu đọc từ thư viện ảnh có dạng (N, H, W, 3). Code dưới chuyển sang tensor cho PyTorch bằng `reshape` — mô hình vẫn train, không báo lỗi, nhưng accuracy thấp bất thường. Tìm lỗi, viết `to_chw(X)` đúng, và so sánh accuracy.

In [ ]:
X7, y7 = make_shapes(2000, seed=5, rgb=True)
X7_test, y7_test = make_shapes(1000, seed=6, rgb=True)


def to_chw_buggy(X):
    return torch.tensor(X).reshape(len(X), 3, X.shape[1], X.shape[2])


def to_chw(X):
    ### START CODE HERE ### (≈ 1 dòng)
    return torch.tensor(X)
    ### END CODE HERE ###


acc7 = {}
for name, conv in [("reshape (lỗi)", to_chw_buggy), ("permute (đúng)", to_chw)]:
    model7 = train(lambda: make_cnn(3), conv(X7), torch.tensor(y7))
    acc7[name] = accuracy(model7, conv(X7_test), torch.tensor(y7_test))
    print(f"{name:15s}: accuracy = {acc7[name]:.3f}")

In [ ]:
check_close(to_chw(X7[:2])[1, 2, 5, 7], X7[1, 5, 7, 2], message="kênh 2 tại (5, 7) đúng vị trí")
check_equal(tuple(to_chw(X7[:2]).shape), (2, 3, 28, 28), "shape CHW")
check_greater(acc7["permute (đúng)"], acc7["reshape (lỗi)"] + 0.1, "sửa lỗi cải thiện rõ")

## L5 — Phân tích

**Bài 8.** **Augmentation giúp bao nhiêu khi dữ liệu tăng?** Với $N \in \{100, 300, 1000, 3000\}$ ảnh train (lấy $N$ ảnh đầu của `make_shapes(3000, seed=0)`), train CNN 1 000 bước có và không có `random_shift_flip`, 2 seed mỗi cấu hình; lưu `gain8[N]` = accuracy(có aug) − accuracy(không aug) trên 1 000 ảnh test. Nhận xét.

In [ ]:
X8, y8 = make_shapes(3000, seed=0)
X8_test, y8_test = make_shapes(1000, seed=1)
X8, X8_test, y8, y8_test = to_tensor(X8), to_tensor(X8_test), torch.tensor(y8), torch.tensor(y8_test)
gain8 = {}
### START CODE HERE ### (≈ 6 dòng)
# với mỗi N: trung bình accuracy 2 seed khi augment=None và augment=random_shift_flip
pass
### END CODE HERE ###

In [ ]:
check_greater(gain8[100], 0.2, "ít dữ liệu: augmentation giúp rất nhiều")
check_greater(gain8[100], gain8[3000] + 0.15, "lợi ích giảm khi dữ liệu tăng")
check_less(abs(gain8[3000]), 0.05, "đủ dữ liệu: gần như không còn lợi ích")

## L6 — Thực tế

**Bài 9.** **Kiểm thử độ bền (robustness).** Trước khi triển khai, kiểm tra mô hình dưới các **nhiễu loạn** thường gặp. Viết `corrupt(x, kind, g)` với: `"sạch"`, `"nhiễu 0.2"` / `"nhiễu 0.4"` (cộng Gaussian độ lệch chuẩn 0.2 / 0.4, cắt về [0, 1]), `"mờ"` (trung bình 3×3), `"tối"` (×0.5), `"tương phản thấp"` ($0.5 + 0.4(x - 0.5)$), `"đảo màu"` ($1 - x$). Train hai mô hình trên 3 000 ảnh (1 500 bước): (a) không augmentation, (b) với `robust_augment` (cho sẵn: nhiễu, mờ, đổi sáng/tương phản ngẫu nhiên). Lưu `report9[tên mô hình][kind]` = accuracy trên 1 000 ảnh test đã nhiễu loạn.

In [ ]:
def robust_augment(x, g):
    B = x.shape[0]
    x = x + torch.rand(B, 1, 1, 1, generator=g) * 0.4 * torch.randn(x.shape, generator=g)
    blur = (torch.rand(B, 1, 1, 1, generator=g) < 0.3).float()
    x = blur * F.avg_pool2d(x, 3, stride=1, padding=1) + (1 - blur) * x
    contrast = 0.3 + 0.7 * torch.rand(B, 1, 1, 1, generator=g)
    brightness = 0.5 + 0.5 * torch.rand(B, 1, 1, 1, generator=g)
    return ((0.5 + (x - 0.5) * contrast) * brightness).clamp(0, 1)


KINDS = ["sạch", "nhiễu 0.2", "nhiễu 0.4", "mờ", "tối", "tương phản thấp", "đảo màu"]


def corrupt(x, kind, g):
    ### START CODE HERE ### (≈ 14 dòng)
    return x
    ### END CODE HERE ###


report9 = {}
for name, aug in [("không augmentation", None), ("robust augmentation", robust_augment)]:
    model9 = train(make_cnn, X8, y8, steps=1500, augment=aug)
    g9 = torch.Generator().manual_seed(0)
    report9[name] = {k: accuracy(model9, corrupt(X8_test, k, g9), y8_test) for k in KINDS}
print(f"{'':20s}" + "".join(f"{k:>16s}" for k in KINDS))
for name, row in report9.items():
    print(f"{name:20s}" + "".join(f"{row[k]:16.3f}" for k in KINDS))

In [ ]:
base9, rob9 = report9["không augmentation"], report9["robust augmentation"]
check_less(base9["nhiễu 0.4"], base9["sạch"] - 0.4, "mô hình thường sụp đổ dưới nhiễu mạnh")
for k in ["nhiễu 0.4", "mờ", "tương phản thấp"]:
    check_greater(rob9[k], base9[k] + 0.2, f"augmentation bền vững hơn: {k}")
check_less(rob9["đảo màu"], 0.5, "không bền với nhiễu loạn KHÔNG có trong augmentation (đảo màu)")

## L7 — Nâng cao

**Bài 10.** **IoU loss vs GIoU loss.** Huấn luyện hộp bằng $1 - \text{IoU}$ có vấn đề: khi hộp dự đoán **không chồng** hộp thật, IoU = 0 với mọi vị trí lân cận → gradient bằng 0. **GIoU** (Rezatofighi et al., 2019): $\text{GIoU} = \text{IoU} - \frac{|C| - |A\cup B|}{|C|}$ với $C$ là hộp nhỏ nhất bao cả hai — vẫn "kéo" hộp về phía đích khi chưa chồng nhau. Viết `iou_and_giou(p, t)` (tensor PyTorch khả vi, xyxy), rồi tối ưu hộp $p_0 = [0, 0, 8, 8]$ về đích $t = [20, 20, 30, 30]$ bằng Adam (lr 0.5, 300 bước) với từng loss; lưu `final_iou10[loss]` = IoU cuối cùng.

In [ ]:
def iou_and_giou(p, t):
    ### START CODE HERE ### (≈ 10 dòng)
    return None, None
    ### END CODE HERE ###


target10 = torch.tensor([20.0, 20.0, 30.0, 30.0])
final_iou10, first_grad10 = {}, {}
for loss_name in ["IoU", "GIoU"]:
    p = torch.tensor([0.0, 0.0, 8.0, 8.0], requires_grad=True)
    opt = torch.optim.Adam([p], lr=0.5)
    for step in range(300):
        opt.zero_grad()
        iou, giou = iou_and_giou(p, target10)
        (1 - (iou if loss_name == "IoU" else giou)).backward()
        if step == 0:
            first_grad10[loss_name] = p.grad.abs().sum().item()
        opt.step()
    final_iou10[loss_name] = iou_and_giou(p.detach(), target10)[0].item()
    print(f"loss {loss_name:4s}: |gradient| bước đầu = {first_grad10[loss_name]:.4f} | hộp cuối = {p.detach().numpy().round(2)} | IoU cuối = {final_iou10[loss_name]:.3f}")

In [ ]:
iou_c, giou_c = iou_and_giou(torch.tensor([10.0, 10.0, 30.0, 30.0]), torch.tensor([20.0, 15.0, 40.0, 35.0]))
check_close(iou_c.item(), 150 / 650, message="IoU khớp câu 3")
check_close(giou_c.item(), 150 / 650 - (750 - 650) / 750, message="GIoU (hộp bao 30×25 = 750)")
check_close(first_grad10["IoU"], 0.0, message="IoU loss: gradient = 0 khi không chồng nhau")
check_greater(first_grad10["GIoU"], 0.0, "GIoU loss: vẫn có gradient")
check_less(final_iou10["IoU"], 0.01, "IoU loss: hộp đứng yên")
check_greater(final_iou10["GIoU"], 0.9, "GIoU loss: hộp tới đích")